# Table of Contents
- 1. Common Table Expressions (CTEs) for Readability
- 2. Multi-Level Nested CTEs vs Temporary Tables
- 3. Recursive CTEs for Hierarchies and Paths
- 4. Performance Optimization Principles in SQL

In [1]:
# Global Imports and Runtime Visual Configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

# Visual styling settings
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10

print("Global libraries for CTEs and recursive queries loaded successfully.")

Global libraries for CTEs and recursive queries loaded successfully.


In [2]:
## 1. Common Table Expressions (CTEs) for Readability

### 1. Definition
# A Common Table Expression (CTE) defines a temporary named result set using a `WITH` clause, executing within the scope of a single query to replace complex nested subqueries and improve readability.

conn = sqlite3.connect(':memory:')
df_emp = pd.DataFrame({
    'department': ['IT', 'IT', 'HR', 'HR'],
    'name': ['Alice', 'Bob', 'Charlie', 'Dave'],
    'salary': [90000, 80000, 60000, 50000]
})
df_emp.to_sql('employees', conn, index=False, if_exists='replace')

query_cte = '''
    WITH DeptAvg AS (
        SELECT department, AVG(salary) as avg_sal
        FROM employees
        GROUP BY department
    )
    SELECT e.name, e.department, e.salary, d.avg_sal
    FROM employees e
    JOIN DeptAvg d ON e.department = d.department
    WHERE e.salary > d.avg_sal
'''
res_cte = pd.read_sql(query_cte, conn)
print(
    "Employees Earning Above Department Average:\n",
    res_cte,
)



Employees Earning Above Department Average:
       name department  salary  avg_sal
0    Alice         IT   90000  85000.0
1  Charlie         HR   60000  55000.0


In [3]:
## 2. Multi-Level Nested CTEs vs Temporary Tables

query_multicte = '''
    WITH Step1 AS (
        SELECT department, salary FROM employees
    ),
    Step2 AS (
        SELECT department, AVG(salary) as mean_sal FROM Step1 GROUP BY department
    )
    SELECT * FROM Step2
'''
res_multi = pd.read_sql(query_multicte, conn)
print("Multi-Level CTE Results:\n", res_multi)


Multi-Level CTE Results:
   department  mean_sal
0         HR   55000.0
1         IT   85000.0


In [4]:
## 3. Recursive CTEs for Hierarchies and Paths
df_org = pd.DataFrame({
    'employee_id': [1, 2, 3],
    'manager_id': [None, 1, 2],
    'name': ['CEO Alice', 'Manager Bob', 'Analyst Charlie']
})
df_org.to_sql('org', conn, index=False, if_exists='replace')

# Recursive CTE query generating a sequence
query_rec = '''
    WITH RECURSIVE NumberSeq(n) AS (
        SELECT 1
        UNION ALL
        SELECT n + 1 FROM NumberSeq WHERE n < 5
    )
    SELECT * FROM NumberSeq
'''
res_rec = pd.read_sql(query_rec, conn)
print("Recursive Sequence Result:\n", res_rec)

Recursive Sequence Result:
    n
0  1
1  2
2  3
3  4
4  5


In [5]:
## 4. Performance Optimization Principles in SQL
query_explain = 'EXPLAIN QUERY PLAN SELECT * FROM employees WHERE salary > 70000'
res_explain = pd.read_sql(query_explain, conn)
print("Query Execution Plan:\n", res_explain)

Query Execution Plan:
    id  parent  notused          detail
0   2       0        0  SCAN employees
